
<div style="float:right ;">
  <img src="https://raw.githubusercontent.com/databricks/koalas/master/Koalas-logo.png" width="180"/>
</div>

# Scale pandas API with Databricks runtime as backend
Using Databricks, Data Scientist don't have to learn a new API to analyse data and deploy new model in production

* If you model is small and fit in a single node, you can use a Single Node cluster with pandas directly
* If your data grow, no need to re-write the code. Just switch to pandas on spark and your cluster will parallelize your compute out of the box.

## Scalability beyond a single machine

<div style="float:right ;">
  <img src="https://databricks.com/fr/wp-content/uploads/2021/09/Pandas-API-on-Upcoming-Apache-Spark-3.2-blog-img-4.png" width="300"/>
</div>


One of the known limitations in pandas is that it does not scale with your data volume linearly due to single-machine processing. For example, pandas fails with out-of-memory if it attempts to read a dataset that is larger than the memory available in a single machine.


Pandas API on Spark overcomes the limitation, enabling users to work with large datasets by leveraging Spark while using Pandas API!


**As result, Data Scientists can access dataset in Unity Catalog with simple SQL or spark command, and then switch to the API they know (pandas) best without having to worry about the table size and scalability!**

*Note: Starting with spark 3.2, pandas API are directly part of spark runtime, no need to import external library!*

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=984752964297111&notebook=%2F01-pyspark-pandas-api-koalas&demo_name=pandas-on-spark&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fpandas-on-spark%2F01-pyspark-pandas-api-koalas&version=1">

In [0]:
%run ./_resources/00-setup $reset_all_data=false

## Loading data from files at scale using pandas API

In [0]:
#instead of Pandas read_json (from pandas import read_json)
#just import pyspark pandas to leverage spark distributed capabilities:
from pyspark.pandas import read_json
print(f"Reading content from Unity Catalog Volume {volume_path}")
pdf = read_json(volume_path+"/users")
print(f"pdf is of type {type(pdf)}")
display(pdf)

In [0]:
import pandas as pd
import numpy as np
from pyspark.pandas import from_pandas
dates = pd.date_range('20130101', periods=6)
df = pd.DataFrame(np.random.randn(6, 4), index=dates, columns=list('ABCD'))
#Transform our pandas dataframe to pandas on spark & have all spark speed & parallelism
pdf = from_pandas(df)
print(f"pdf is of type {type(pdf)}")
#Apply standard pandas operationhttps://e2-demo-field-eng.cloud.databricks.com/?o=1444828305810485#
pdf.mean()

In [0]:
dbutils.fs.ls(volume_path+"/users")

In [0]:
# Spark dataframe can also be switched to a Pandas dataframe with one instruction 
# As example, You could easily read one table from Unity Catalog, then leverage pandas API for your Data Science analysis
users = spark.read.json(volume_path+"/users").pandas_api()
print(f"users is of type {type(users)}")
print(users.describe())

type(users)

## Pandas APIs are now fully available to explore our dataset
Existing pandas code will now work and scale out of the box:

In [0]:
users["age_group"].value_counts(dropna=False).sort_values()

## Running SQL on top of pandas dataframe
Pandas dataframe on spark can be queried using plain SQL, allowing a perfect match between pandas python API and SQL usage

In [0]:
from pyspark.pandas import sql
age_group = 2
#Note: this might not work as expected on serverless compute - stay tune as we're workign on it
sql("""SELECT age_group, COUNT(*) AS customer_per_segment FROM {users} 
        where age_group > {age_group}
      GROUP BY age_group ORDER BY age_group """, users=users, age_group=age_group)

## Pandas visualisation
Pandas api on spark use plotly with interactive charts. Use the standard `.plot` on a pandas dataframe to get insights: 

In [0]:
df = users.groupby('gender')['age_group'].value_counts().unstack(0)
df.plot.bar(title="Customer age distribution")


That's it! you're now ready to use all the capabilities of Databricks while staying on the API you know the best!

## Going further with koalas

Want to know more? Check the [koalas documentation](https://koalas.readthedocs.io/en/latest/) and the integration with [spark 3.2](https://databricks.com/fr/blog/2021/10/04/pandas-api-on-upcoming-apache-spark-3-2.html)